# Induction results table

Regenerates the paper's induction accuracy table (`tab:induction-results`) from the
public release bucket `s3://smolbench-public-release` alone. It needs no AWS credentials
and no local results tree: every request is anonymous, and the bucket's `induction/`
prefix holds exactly the 1,440 result files the table was computed from
(16 models x 3 conditions x 30 seeds).

Run it with the repo venv (`uv sync --all-extras`, then pick the `.venv` kernel). It
downloads about 1.1 GB.

## What the table reports

Each result file is one replicate: one seed's quiz of 9 marks for one model under one
condition. The file layout is the private results store's, unchanged:
`induction/<model>/seed=<seed>/<condition>--<UTC run stamp>.yaml`.

| Column | Condition key | Prompt |
| --- | --- | --- |
| high density | `intens` | the rule stated intensionally |
| high density + irrelevant | `noise_intens` | the same prompt, padded with whitespace to the `extens` prompt's exact token count under the model's own tokenizer |
| low density | `extens` | the rule given extensionally, as a labelled listing |

- **Replicate accuracy** = marks with a truthy `score`, divided by 9. A `null` score
  (an answer that could not be graded) counts as wrong, never as missing.
- **Cell** = mean over the 30 replicates, ± a spread built from their *sample*
  standard deviation (ddof = 1). The table is printed three times, with ± as 1 SD
  (the published table), 2 SD, and the 95% confidence interval for the mean.
- **Bold** marks the row's highest mean. Ties are all bolded.
- **Δ** = 100 x (rounded mean − rounded low-density mean), in points, using the
  3-decimal means as printed. Using the unrounded means moves 7 of the 32 Δ cells by 0.1.
- **One run per replicate.** The private store is append-only, and a replicate that was
  re-collected keeps its earliest run (the study's pass@1 rule). Only those earliest
  runs were published, so the selection below should find exactly one file per
  (model, seed, condition), and it asserts that.

In [1]:
import concurrent.futures
import statistics

import boto3
import yaml
from botocore import UNSIGNED
from botocore.config import Config

try:  # libyaml parses the ~0.75 MB result files several times faster
    from yaml import CSafeLoader as Loader
except ImportError:
    from yaml import SafeLoader as Loader

BUCKET = "smolbench-public-release"
REGION = "us-west-2"
PREFIX = "induction"
SEEDS = range(30)
MARKS_PER_REPLICATE = 9

#: Table column order -> condition key in the store.
ARMS = {"high": "intens", "irrelevant": "noise_intens", "low": "extens"}

#: (store model id, row label, rule before the row). Rows appear in table order; a
#: rule separates model families.
LANES = [
    ("gemma-4-e2b", "Gemma4 E2B-it", None),
    ("gemma-4-12b", "Gemma4 12B-it", None),
    ("gemma-4-31b", "Gemma4 31B-it", None),
    ("nemotron-3-nano-4b", "Nemotron3 Nano-4B", r"\addlinespace"),
    ("nemotron-3-nano-30b-a3b", "Nemotron3 Nano-30B", None),
    ("nemotron-3-super-120b-a12b", "Nemotron3 Super-120B", None),
    ("qwen3.5-27b", "Qwen3.5 27B", r"\addlinespace"),
    ("qwen3.5-122b-a10b", "Qwen3.5 122B", None),
    ("qwen3.5-397b-a17b", "Qwen3.5 397B", None),
    ("deepseek-v4-flash", "Deepseek V4-Flash", r"\addlinespace"),
    ("deepseek-v3.1", "Deepseek V3.1", None),
    ("glm-4.7-flash", "GLM-4.7-Flash", r"\addlinespace"),
    ("glm-4.7", "GLM-4.7", None),
    ("ministral-3-3b", "Ministral3-2512 3B", r"\addlinespace"),
    ("ministral-3-8b", "Ministral3-2512 8B", None),
    ("ministral-3-14b", "Ministral3-2512 14B", None),
]

s3 = boto3.client(
    "s3",
    region_name=REGION,
    config=Config(signature_version=UNSIGNED, max_pool_connections=32),
)

## Select one result file per (model, seed, condition)

In [2]:
def select_keys(model):
    """Map (seed, condition) -> key of the earliest run for one model.

    Parameters
    ----------
    model : str
        Store model id, e.g. ``"gemma-4-e2b"``.

    Returns
    -------
    dict[tuple[int, str], str]
        One key per (seed, condition) for the table's three conditions.
    """
    runs = {}
    pages = s3.get_paginator("list_objects_v2").paginate(
        Bucket=BUCKET, Prefix=f"{PREFIX}/{model}/"
    )
    for page in pages:
        for obj in page.get("Contents", []):
            seed_part, name = obj["Key"].split("/")[-2:]
            condition, stamp = name.removesuffix(".yaml").split("--", 1)
            seed = int(seed_part.removeprefix("seed="))
            runs.setdefault((seed, condition), []).append((stamp, obj["Key"]))
    for address, found in runs.items():
        assert len(found) == 1, f"{model} {address}: expected one run, found {found}"
    wanted = {(seed, cond) for seed in SEEDS for cond in ARMS.values()}
    assert set(runs) == wanted, f"{model}: missing {sorted(wanted - set(runs))}"
    return {address: min(found)[1] for address, found in runs.items()}


keys = {model: select_keys(model) for model, _, _ in LANES}
print(sum(len(v) for v in keys.values()), "result files selected")

1440 result files selected


## Download and score every replicate

In [3]:
def replicate_accuracy(key):
    """Fraction of one replicate's marks scored correct; a null score is wrong."""
    body = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
    marks = yaml.load(body, Loader=Loader)["marks"]
    assert len(marks) == MARKS_PER_REPLICATE, key
    return sum(bool(mark["score"]) for mark in marks) / MARKS_PER_REPLICATE


jobs = [
    (model, cond, seed, key)
    for model, by_address in keys.items()
    for (seed, cond), key in by_address.items()
]
with concurrent.futures.ThreadPoolExecutor(max_workers=32) as pool:
    scores = pool.map(replicate_accuracy, [key for *_, key in jobs])
    accuracy = {(m, c, s): a for (m, c, s, _), a in zip(jobs, scores)}
print(len(accuracy), "replicates scored")

1440 replicates scored


## Build the table three ways

The means, bold marks and Δ columns are the same in all three; only the ± changes.

| Version | ± | Label |
| --- | --- | --- |
| 1 SD | sample SD of the 30 replicate accuracies (the published table) | `tab:induction-results` |
| 2 SD | twice that | `tab:induction-results-2sd` |
| 95% CI | half-width of the 95% t confidence interval for the mean, 2.045 x SD / sqrt(30) | `tab:induction-results-ci95` |

The SD versions describe how much a single seed's accuracy varies. The CI describes how
precisely 30 seeds pin down the mean, so it is about 0.37 SD wide.

In [4]:
T_975_DF29 = 2.045  # two-sided 95% t quantile, 29 degrees of freedom (30 replicates)

SPREADS = {
    "1 SD": (
        lambda sd: sd,
        "tab:induction-results",
        r"$\pm$ one sample standard deviation across seeds",
    ),
    "2 SD": (
        lambda sd: 2 * sd,
        "tab:induction-results-2sd",
        r"$\pm$ two sample standard deviations across seeds",
    ),
    "95% CI": (
        lambda sd: T_975_DF29 * sd / len(SEEDS) ** 0.5,
        "tab:induction-results-ci95",
        r"$\pm$ the half-width of the $95\%$ $t$ confidence interval for the mean",
    ),
}


def cell_stats(model, cond):
    """Mean and sample SD (ddof=1) of one cell's 30 replicate accuracies."""
    values = [accuracy[(model, cond, seed)] for seed in SEEDS]
    return statistics.mean(values), statistics.stdev(values)


def fmt_delta(points):
    """One-decimal Δ, with a negative wrapped in math mode for the minus sign."""
    return f"{points:.1f}" if points >= 0 else f"$-{abs(points):.1f}$"


stats = {
    model: {col: cell_stats(model, cond) for col, cond in ARMS.items()}
    for model, _, _ in LANES
}

HEADER = r"""\begin{table}[t]
    \centering
    \footnotesize
    LLM Induction Accuracy
    \begin{tabular}{lcccrr}
        \toprule
        Model
            & \makecell{high density}
            & \makecell{high density \\ $+$ irrelevant}
            & \makecell{low density}
            & \makecell{$\Delta$ (high \\ $-$ low)}
            & \makecell{$\Delta$ (irrelevant \\ $-$ low)} \\
        \hline"""
CAPTION = (
    "For induction, LLMs with high density information generally outperform low "
    "density information, even when context length is accounted for by appending "
    "irrelevant information to high density information. Each cell is the mean "
    "accuracy over 30 seeds {spread}."
)


def render_table(spread, label, spread_text):
    """LaTeX for the whole table with `spread(sd)` as each cell's ±."""
    rows = []
    for model, name, rule in LANES:
        s = stats[model]
        best = max(mean for mean, _ in s.values())
        cells = [name]
        for col in ARMS:
            mean, sd = s[col]
            body = f"{mean:.3f} \\pm {spread(sd):.3f}"
            cells.append(f"$\\mathbf{{{body}}}$" if mean == best else f"${body}$")
        low = round(s["low"][0], 3)
        for col in ("high", "irrelevant"):
            cells.append(fmt_delta(100 * (round(s[col][0], 3) - low)))
        rows.append((rule, cells))

    widths = [max(len(cells[i]) for _, cells in rows) for i in range(6)]
    lines = [HEADER]
    for rule, cells in rows:
        if rule:
            lines.append(f"        {rule}")
        padded = [cells[0].ljust(widths[0])]
        padded += [c.rjust(w) for c, w in zip(cells[1:], widths[1:])]
        lines.append("        " + " & ".join(padded) + r" \\")
    lines += [
        r"        \bottomrule",
        r"    \end{tabular}",
        f"    \\caption{{{CAPTION.format(spread=spread_text)}}}",
        f"    \\label{{{label}}}",
        r"\end{table}",
    ]
    return "\n".join(lines)


tables = {name: render_table(*spec) for name, spec in SPREADS.items()}
for name, table in tables.items():
    print(f"% ---- ± = {name} ----")
    print(table)
    print()

% ---- ± = 1 SD ----
\begin{table}[t]
    \centering
    \footnotesize
    LLM Induction Accuracy
    \begin{tabular}{lcccrr}
        \toprule
        Model
            & \makecell{high density}
            & \makecell{high density \\ $+$ irrelevant}
            & \makecell{low density}
            & \makecell{$\Delta$ (high \\ $-$ low)}
            & \makecell{$\Delta$ (irrelevant \\ $-$ low)} \\
        \hline
        Gemma4 E2B-it        &          $0.948 \pm 0.100$ & $\mathbf{0.981 \pm 0.042}$ &          $0.115 \pm 0.020$ &    83.3 &    86.6 \\
        Gemma4 12B-it        & $\mathbf{1.000 \pm 0.000}$ &          $0.993 \pm 0.028$ &          $0.548 \pm 0.162$ &    45.2 &    44.5 \\
        Gemma4 31B-it        & $\mathbf{1.000 \pm 0.000}$ & $\mathbf{1.000 \pm 0.000}$ &          $0.985 \pm 0.048$ &     1.5 &     1.5 \\
        \addlinespace
        Nemotron3 Nano-4B    &          $0.633 \pm 0.202$ & $\mathbf{0.719 \pm 0.186}$ &          $0.396 \pm 0.162$ &    23.7 &    32.3 \\
      

## Check against the published table

These are the paper's printed values (mean, ± per column, then the two Δ cells). The
published ± is 1 SD, so this checks the 1 SD version. Every cell must match exactly
after formatting.

In [5]:
PUBLISHED = {
    "gemma-4-e2b": ((0.948, 0.100), (0.981, 0.042), (0.115, 0.020), 83.3, 86.6),
    "gemma-4-12b": ((1.000, 0.000), (0.993, 0.028), (0.548, 0.162), 45.2, 44.5),
    "gemma-4-31b": ((1.000, 0.000), (1.000, 0.000), (0.985, 0.048), 1.5, 1.5),
    "nemotron-3-nano-4b": ((0.633, 0.202), (0.719, 0.186), (0.396, 0.162), 23.7, 32.3),
    "nemotron-3-nano-30b-a3b": ((0.956, 0.069), (0.937, 0.070), (0.711, 0.150), 24.5, 22.6),
    "nemotron-3-super-120b-a12b": ((1.000, 0.000), (0.996, 0.020), (0.841, 0.126), 15.9, 15.5),
    "qwen3.5-27b": ((1.000, 0.000), (1.000, 0.000), (0.974, 0.056), 2.6, 2.6),
    "qwen3.5-122b-a10b": ((0.996, 0.020), (0.993, 0.028), (0.974, 0.056), 2.2, 1.9),
    "qwen3.5-397b-a17b": ((1.000, 0.000), (1.000, 0.000), (0.974, 0.048), 2.6, 2.6),
    "deepseek-v4-flash": ((0.959, 0.068), (0.993, 0.028), (0.963, 0.067), -0.4, 3.0),
    "deepseek-v3.1": ((1.000, 0.000), (1.000, 0.000), (0.993, 0.028), 0.7, 0.7),
    "glm-4.7-flash": ((0.289, 0.150), (0.122, 0.144), (0.393, 0.133), -10.4, -27.1),
    "glm-4.7": ((0.959, 0.062), (0.789, 0.236), (0.889, 0.065), 7.0, -10.0),
    "ministral-3-3b": ((0.663, 0.197), (0.637, 0.214), (0.407, 0.205), 25.6, 23.0),
    "ministral-3-8b": ((0.789, 0.236), (0.215, 0.198), (0.222, 0.120), 56.7, -0.7),
    "ministral-3-14b": ((0.470, 0.181), (0.026, 0.048), (0.081, 0.092), 38.9, -5.5),
}
mismatches = []
for model, (*published_cells, d_high, d_irr) in PUBLISHED.items():
    got = [(f"{m:.3f}", f"{sd:.3f}") for m, sd in stats[model].values()]
    want = [(f"{m:.3f}", f"{sd:.3f}") for m, sd in published_cells]
    low = round(stats[model]["low"][0], 3)
    got_d = [f"{100 * (round(stats[model][c][0], 3) - low):.1f}" for c in ("high", "irrelevant")]
    want_d = [f"{d_high:.1f}", f"{d_irr:.1f}"]
    if got != want or got_d != want_d:
        mismatches.append((model, got, got_d, want, want_d))
assert not mismatches, mismatches
print(f"all {len(PUBLISHED) * 5} cells match the published table")

all 80 cells match the published table
